# Reporte de Proyecto Final de Robótica: Manipulador RRR

## 1. Introducción
El presente documento detalla el desarrollo, análisis matemático y simulación de un brazo manipulador de 3 grados de libertad con configuración RRR. El proyecto integra el modelado paramétrico del robot mediante archivos URDF, la formulación de su cinemática directa y el desarrollo de un controlador de trayectoria numérico basado en la pseudoinversa del Jacobiano para la simulación interactiva en RViz.

## 2. Descripción del Modelo URDF y Entorno RViz
La geometría física del brazo robótico fue descrita mediante un archivo paramétrico URDF (`robot.urdf`), definiendo cada eslabón y acoplándolos mediante articulaciones tipo `revolute`. 

Para la visualización interactiva, se desarrolló el archivo de lanzamiento `rviz_bringup.launch.py`, el cual ejecuta los siguientes nodos clave:
* **`robot_state_publisher`:** Lee el URDF y publica las transformaciones estáticas de los eslabones.
* **`rviz2`:** Renderiza la simulación 3D utilizando mallas geométricas y configuraciones previas.
* **`joint_state_publisher_gui`:** Configurado para escuchar el tópico `/joint_states_goals`, actuando como el puente de comunicación entre el cálculo matemático y el entorno gráfico.

<div align="center">
  <img src="Imagenes/rviz1.png" alt="Primer despliegue en RViz" width="550" height="auto" display="block"/>
</div>

### 2.1. Correspondencia entre Modelo Físico y Matemático
Para garantizar la precisión de la simulación, los parámetros geométricos definidos en el archivo `robot.urdf` mantienen una equivalencia directa con el modelo cinemático implementado en Python. Las longitudes de los eslabones utilizadas en la clase `Robot` (definidas por defecto como $L_1=0.2\text{m}, L_2=0.3\text{m}, L_3=0.5\text{m}$) corresponden a las dimensiones físicas de los cilindros y cajas configurados en las etiquetas `<geometry>` del URDF.

Esta vinculación es fundamental para que las transformaciones calculadas por la cinemática directa e inversa coincidan con la posición visual del robot en RViz, permitiendo una validación precisa del comportamiento del sistema.

In [ ]:
<robot name="RobotRRR">
  <link name="base_link">
    <visual>
      <geometry>
        <cylinder radius="0.2" length="0.1"/>
      </geometry>
      <origin xyz="0 0 -0.05"/>
      <material name="black">
        <color rgba="0 0 0 0.5"/>
      </material>
    </visual>
  </link>

  <link name="body_link">
    <visual>
      <geometry>
        <box size="0.05 0.05 0.2"/>
      </geometry>
      <origin xyz="0 0 0.1"/>
      <material name="cyan">
        <color rgba="0 1 1 1"/>
      </material>
    </visual>
  </link>

  <link name="shoulder_link">
    <visual>
      <geometry>
        <box size="0.3 0.05 0.05"/>
      </geometry>
      <origin xyz="0.15 0 0"/>
      <material name="magenta">
        <color rgba="1 0 1 1"/>
      </material>
    </visual>
  </link>

  <link name="arm_link">
    <visual>
      <geometry>
        <box size="0.5 0.05 0.05"/>
      </geometry>
      <origin xyz="0.25 0 0"/>
      <material name="yellow">
        <color rgba="1 1 0 1"/>
      </material>
    </visual>
  </link>

  <joint name="shoulder_joint" type="revolute">
    <parent link="base_link"/>
    <child link="body_link"/>
    <axis xyz="0 0 1"/> <limit lower="-3.14" upper="3.14" effort="10.0" velocity="10.0"/>
    <origin xyz="0 0 0" rpy="0 0 0"/> </joint>

  <joint name="arm_joint" type="revolute">
    <parent link="body_link"/>
    <child link="shoulder_link"/>
    <axis xyz="0 0 1"/> <limit lower="-3.14" upper="3.14" effort="10.0" velocity="10.0"/>
    <origin xyz="0 0 0.2" rpy="1.57 0 0"/> </joint>

  <joint name="forearm_joint" type="revolute">
    <parent link="shoulder_link"/>
    <child link="arm_link"/>
    <axis xyz="0 0 1"/> <limit lower="-3.14" upper="3.14" effort="10.0" velocity="10.0"/>
    <origin xyz="0.3 0 0" rpy="0 0 0"/> </joint>
</robot>

## 3. Análisis Matemático: Cinemática Directa
El modelo de cinemática directa permite obtener la posición exacta del efector final a partir de los ángulos articulares $(\theta_1, \theta_2, \theta_3)$. 

### Diagrama de Eslabones y Sistemas de Referencia
El siguiente diagrama muestra la disposición de los eslabones y la orientación de los ejes coordenados (X, Y, Z) definidos para el manipulador, los cuales fueron la base para la construcción de las matrices de rotación:

<div align="center">
  <img src="Imagenes/xyz.png" alt="Diagrama de eslabones y ejes" width="550" height="auto" display="block"/>
</div>

### Matrices de Transformación y Jacobiano
La postura final $\xi$ se obtiene del producto de las matrices de transformación homogénea de cada junta, siguiendo la secuencia:
$$^0T_p = ^0T_1(\theta_1) \cdot ^1T_2(\theta_2) \cdot ^2T_3(\theta_3) \cdot ^3T_p$$

In [ ]:
from sympy import *
import matplotlib.pyplot as plt
import numpy as np 

class Robot():
  def __init__(self, l:tuple[float]=(0.2, 0.3, 0.5)):
    th1, th2, th3 = symbols("theta_1,theta_2,theta_3")

    # 1. Transformaciones 3D 
    T_0_1 = self.tr_h(alpha=th1)

    T_1_2 = self.tr_h(z=l[0]) \
           * self.tr_h(gamma=pi/2) \
           * self.tr_h(alpha=th2)

    T_2_3 = self.tr_h(x=l[1]) \
           * self.tr_h(alpha=th3)

    T_3_p = self.tr_h(x=l[2])

    T_0_p = T_0_1 * T_1_2 * T_2_3 * T_3_p
    T_0_p = simplify(T_0_p)
    
    # 2. Vector de postura 3D (X, Y, Z)
    xi_0_p = Matrix([T_0_p[0, 3],
                     T_0_p[1, 3],
                     T_0_p[2, 3]])
                     
    # 3. Jacobiano 3x3
    J = xi_0_p.jacobian(Matrix([th1, th2, th3]))

    # Velocidades espaciales deseadas
    x_dot, y_dot, z_dot = symbols("x_dot, y_dot, z_dot")
    t = symbols("t")
    a_0, a_1, a_2, a_3, a_4, a_5 = symbols("a_0, a_1, a_2, a_3, a_4, a_5")
    lam = a_0 + a_1 * t + a_2 * t**2 + a_3 * t**3 + a_4 * t**4 + a_5 * t**5    
    lam_dot = diff(lam, t)
    lam_dot_dot = diff(lam_dot, t)
    
    # Almacenar variables
    self.th1, self.th2, self.th3 = th1, th2, th3
    self.xi_0_p = xi_0_p

    print(
    xi_0_p.subs({
        th1:0,
        th2:0,
        th3:0
    }).evalf()
)


$$ 
J = \begin{bmatrix} 
-(0.3\cos(\theta_2) + 0.5\cos(\theta_2+\theta_3))\sin(\theta_1) & -(0.3\sin(\theta_2) + 0.5\sin(\theta_2+\theta_3))\cos(\theta_1) & -0.5\sin(\theta_2+\theta_3)\cos(\theta_1) \\
(0.3\cos(\theta_2) + 0.5\cos(\theta_2+\theta_3))\cos(\theta_1) & -(0.3\sin(\theta_2) + 0.5\sin(\theta_2+\theta_3))\sin(\theta_1) & -0.5\sin(\theta_2+\theta_3)\sin(\theta_1) \\
0 & 0.3\cos(\theta_2) + 0.5\cos(\theta_2+\theta_3) & 0.5\cos(\theta_2+\theta_3)
\end{bmatrix} 
$$

## 4. Planteamiento de la Trayectoria



In [ ]:
    self.J = J # Guardamos J normal, no la inversa
    self.x_dot, self.y_dot, self.z_dot = x_dot, y_dot, z_dot
    self.a_0, self.a_1, self.a_2, self.a_3, self.a_4, self.a_5 = a_0, a_1, a_2, a_3, a_4, a_5
    self.t = t
    self.lam, self.lam_dot, self.lam_dot_dot = lam, lam_dot, lam_dot_dot

  def def_tray(self, t_f:float=2, frec:float=100, 
               th_i:tuple[float]=(0.1, 0.1, 0.1), 
               xi_f:tuple[float]=(0.6, 0.1, 0)):
    
    # Evaluar a float desde el inicio
    xi_i = self.xi_0_p.subs({self.th1: th_i[0], 
                             self.th2: th_i[1], 
                             self.th3: th_i[2]}).evalf() 
    self.dt = 1.0/frec
    self.muestras = int(t_f * frec + 1)

    eq1 = self.lam.subs({self.t: 0})
    eq2 = self.lam.subs({self.t: t_f}) - 1
    eq3 = self.lam_dot.subs({self.t: 0})
    eq4 = self.lam_dot.subs({self.t: t_f})
    eq5 = self.lam_dot_dot.subs({self.t: 0})
    eq6 = self.lam_dot_dot.subs({self.t: t_f})
    solutions = solve((eq1, eq2, eq3, eq4, eq5, eq6),
                  (self.a_0, self.a_1, self.a_2, self.a_3, self.a_4, self.a_5))
    
    lam_s         = self.lam.subs(solutions)
    lam_dot_s     = self.lam_dot.subs(solutions)
    lam_dot_dot_s = self.lam_dot_dot.subs(solutions)
    
    xi_f_mat = Matrix([xi_f[0], xi_f[1], xi_f[2]])
    xi_eq         = xi_i + (xi_f_mat - xi_i) * lam_s
    xi_dot_eq     = (xi_f_mat - xi_i) * lam_dot_s
    xi_dot_dot_eq = (xi_f_mat - xi_i) * lam_dot_dot_s
    
    t_m = Matrix.zeros(1, self.muestras)
    for i in range(self.muestras):
      t_m[i] = self.dt * i
      
    xi_m         = Matrix.zeros(3, self.muestras)
    xi_dot_m     = Matrix.zeros(3, self.muestras)
    xi_dot_dot_m = Matrix.zeros(3, self.muestras)
    
    for i in range(self.muestras):
      xi_m[:, i]         = xi_eq.subs({self.t: t_m[i]})
      xi_dot_m[:, i]     = xi_dot_eq.subs({self.t: t_m[i]})
      xi_dot_dot_m[:, i] = xi_dot_dot_eq.subs({self.t: t_m[i]})

    th_m         = Matrix.zeros(3, self.muestras)
    th_dot_m     = Matrix.zeros(3, self.muestras)
    th_dot_dot_m = Matrix.zeros(3, self.muestras)
    
    th_m[:, 0] = Matrix([th_i[0], th_i[1], th_i[2]])

## 5. Cinematica Inversa



A partir del modelo de cinemática directa, se deriva la expresión de la cinemática inversa que relaciona las velocidades articulares ($\dot{q}$) con la velocidad del efector final ($\dot{\xi}$) mediante la matriz Jacobiana ($J$):

$$\dot{q} = J^\dagger \dot{\xi}$$

Donde $J^\dagger$ es la pseudoinversa de Moore-Penrose. Dado que este modelo permite obtener únicamente las velocidades, el sistema integra numéricamente estas expresiones para obtener la posición de las juntas ($q$) y sus aceleraciones ($\ddot{q}$) mediante el método de Euler en cada paso de tiempo $dt$, garantizando la continuidad en la trayectoria del manipulador:

* **Posición:** $q_{k+1} = q_k + \dot{q}_k \cdot dt$
* **Aceleración:** $\ddot{q}_k \approx \frac{\dot{q}_k - \dot{q}_{k-1}}{dt}$

In [ ]:
    for i in range(self.muestras):
      # Evaluar Jacobiano en la posición actual
      J_num = self.J.subs({self.th1: th_m[0, i], 
                           self.th2: th_m[1, i], 
                           self.th3: th_m[2, i]}).evalf()
      
      # Convertir a numpy array y calcular pseudoinversa
      J_np = np.array(J_num).astype(np.float64)
      J_inv_np = np.linalg.pinv(J_np) 
      
      # Vector de velocidad deseada actual
      xi_dot_np = np.array([[xi_dot_m[0, i]], 
                            [xi_dot_m[1, i]], 
                            [xi_dot_m[2, i]]], dtype=np.float64)
      
      # th_dot = J_inv * xi_dot
      th_dot_np = np.dot(J_inv_np, xi_dot_np)
      
      # Guardar resultados
      th_dot_m[0, i] = th_dot_np[0, 0]
      th_dot_m[1, i] = th_dot_np[1, 0]
      th_dot_m[2, i] = th_dot_np[2, 0]
      
      if i < self.muestras - 1:
        th_m[:, i+1] = th_m[:, i] + th_dot_m[:, i] * self.dt
      if not (i == 0):
        th_dot_dot_m[:, i-1] = (th_dot_m[:, i] - th_dot_m[:, i-1]) / self.dt
      
    self.xi_m = xi_m
    self.xi_dot_m = xi_dot_m
    self.xi_dot_dot_m = xi_dot_dot_m
    self.th_m = th_m
    self.th_dot_m = th_dot_m
    self.th_dot_dot_m = th_dot_dot_m
    self.t_m = t_m

    xi_real = self.xi_0_p.subs({
    self.th1: self.th_m[0,-1],
    self.th2: self.th_m[1,-1],
    self.th3: self.th_m[2,-1]
    }).evalf()

    print("\nObjetivo:")
    print(xi_f_mat)

    print("\nAlcanzado:")
    print(xi_real)

    print("\nError:")
    print(xi_f_mat - xi_real)

## 6. Aplicación de la cinemática inversa
Finalmente, a partir de los puntos de la trayectoria deseada en el espacio cartesiano y el modelo de cinemática inversa implementado, se calcularon las trayectorias articulares del robot. Este proceso permitió determinar las posiciones ($q$), velocidades ($\dot{q}$) y aceleraciones ($\ddot{q}$) de cada junta en función del tiempo. Los resultados de este análisis se presentan gráficamente a continuación, permitiendo validar la respuesta dinámica del manipulador ante el perfil de movimiento seleccionado.

In [ ]:
def imp_tray(self):
    fig, (x_g, y_g, z_g) = plt.subplots(nrows = 1, ncols = 3)
    fig.suptitle("Posiciones del efector final")
    x_g.set_title("X")
    y_g.set_title("Y")
    z_g.set_title("Z")
    x_g.plot(self.t_m.T,  self.xi_m[0, :].T, color="RED")
    y_g.plot(self.t_m.T,  self.xi_m[1, :].T, color="green")
    z_g.plot(self.t_m.T, self.xi_m[2, :].T, color=(0,0,1))
    plt.show()

  def imp_junt(self):
    fig, (th1_g, th2_g, th3_g) = plt.subplots(nrows = 1, ncols = 3)
    fig.suptitle("Posiciones de las juntas")
    th1_g.set_title("th1")
    th2_g.set_title("th2")
    th3_g.set_title("th3")
    th1_g.plot(self.t_m.T,  self.th_m[0, :].T, color="RED")
    th2_g.plot(self.t_m.T,  self.th_m[1, :].T, color="green")
    th3_g.plot(self.t_m.T,  self.th_m[2, :].T, color=(0,0,1))
    plt.show()

  def tr_h(self, x=0, y=0, z=0, gamma=0, beta=0, alpha=0):
    t_x = Matrix([[1, 0, 0, x], [0, cos(gamma), -sin(gamma), 0], [0, sin(gamma), cos(gamma), 0], [0, 0, 0, 1]])
    t_y = Matrix([[cos(beta), 0, sin(beta), 0], [0, 1, 0, y], [-sin(beta), 0, cos(beta), 0], [0, 0, 0, 1]])
    t_z = Matrix([[cos(alpha), -sin(alpha), 0, 0], [sin(alpha), cos(alpha), 0, 0], [0, 0, 1, z], [0, 0, 0, 1]])
    tr = simplify(t_x * t_y * t_z)
    return tr

def main():
  robot = Robot()
  robot.def_tray()
  robot.imp_tray()
  robot.imp_junt()
if __name__ == "__main__":
  main()

robot = Robot()

robot.def_tray(
    xi_f=(0.6,0.1,0)
)

Las siguientes gráficas ilustran el comportamiento de las articulaciones durante la ejecución:

<div align="center">
  <img src="Imagenes/Posicion.png" alt="Gráficas de trayectoria" width="650"/>
</div>

<div align="center">
  <img src="Imagenes/juntas.png" alt="Gráficas de trayectoria" width="650"/>
</div>

### 7. Conclusiones

La integración entre el modelo físico en URDF y el modelo matemático asegura una correspondencia exacta, garantizando la fidelidad del robot en la simulación. El método de la pseudoinversa de Moore-Penrose resolvió eficientemente la cinemática inversa mediante un enfoque iterativo, validado por un error residual mínimo que demuestra la estabilidad del control. La implementación de un polinomio de quinto orden eliminó discontinuidades, asegurando transiciones suaves en velocidad y aceleración. El análisis de las variables articulares confirmó un comportamiento fluido y predecible, mientras que el uso de integración numérica facilitó la obtención de variables cinemáticas superiores. Finalmente, esta trayectoria optimizada minimiza los esfuerzos en los actuadores, favoreciendo la integridad mecánica y la vida útil del sistema. 
